# 病灶重叠与 FA 审核

从合成 T2、病灶、图谱和 DWI 指标练习三种方法。Notebook 调用与命令行相同的后端。详见 [方法与操作](../docs/LESIONLOAD.md)。 真实病灶练习可选 [ARC 的同访视 T2 与专家标签](../docs/PUBLIC_DATA.md#arc-lesion-exercise)，HCP 用于健康结构或 DWI。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import tempfile

cwd = Path.cwd()
repo = next((p for p in (cwd, *cwd.parents) if (p / "run_lesionload.py").is_file()), None)
if repo is None:
    raise RuntimeError("从 NEO_MRI 检出目录或 notebooks 打开。")
work = repo / "work"
work.mkdir(exist_ok=True)
run = Path(tempfile.mkdtemp(prefix="lesionload_tutorial_", dir=work)) / "synthetic"
subprocess.run([sys.executable, "-B", str(repo / "run_lesionload.py"), "demo",
                "--output", str(run)], cwd=repo, check=True)
results = run / "results"
receipt = json.loads((results / "receipt.json").read_text(encoding="utf-8"))
print(results)

## 先核对分母

N 是参考束体素数，C 是 T2 覆盖数，H 是病灶命中数，U 是未知体素数。先找左束覆盖不足、右束覆盖完整、空束三个情形。

In [ ]:
for method, result in receipt["methods"].items():
    print("\n", method)
    for row in result["summary"]:
        print(row["region"], [row[key] for key in ("atlas_voxels", "covered_voxels",
             "hits_voxels", "unknown_voxels")], row["fraction_bounds"])

## 核对轴位峰值和 Max-WLL

逐层查看 N、C、H，找出已知命中率下界最高的层。对于 HCPA，最大的截面积乘以该峰值。覆盖不全时，上界可能由另一层决定。

In [ ]:
import csv
with (results / "HCPA" / "axial.csv").open(encoding="utf-8", newline="") as stream:
    print([row for row in csv.DictReader(stream) if row["region"] == "Left_CST"])
left = next(row for row in receipt["methods"]["HCPA"]["summary"] if row["region"] == "Left_CST")
print("Max-WLL 区间", left["max_wll_bounds_voxel"])

## 打开 C–F 审核页

移动滑块，查看 3D 轮廓、轴位、覆盖内重叠率与 FA 同步更新。点击 F 图中的横轴位置也可选层。查看 FA 表中的 ROI 数、DWI 有覆盖数和实际有效 FA 数。

In [ ]:
from IPython.display import FileLink, display
for method in receipt["methods"]:
    path = (results / method / "report.html").relative_to(repo)
    display(FileLink(Path("..") / path if cwd == repo / "notebooks" else path))
with (results / "SMATT" / "fa_whole.csv").open(encoding="utf-8", newline="") as stream:
    print([row for row in csv.DictReader(stream) if row["region"] == "Left_M1"])

## 用于真实数据

填好本地配置，再单独运行命令行入口。先确认同访视的 T2、病灶和审核过的 T1/MNI 变换，并检查每个图谱的网格。Boyne 方法 C 的皮层分束来自公开可用掩膜的本地重放。**DLPFC 待解决**。补齐作者左右体积掩膜或确认 Gen6 白质面生成链后，再复核 FEFarea、Other、Unassigned 的排他归属。